# Identifying and Removing Duplicate Records

This notebook demonstrates how to detect, inspect, and remove duplicate records in Pandas DataFrames to improve data quality.

## Part 1: Loading and Inspecting Data with Duplicates

We'll start by loading datasets that contain intentional duplicate records.

In [ ]:
import pandas as pd
import numpy as np

# Load datasets with duplicate records
employees_df = pd.read_csv('../employees_with_duplicates.csv')
sales_df = pd.read_csv('../sales_with_duplicates.csv')

print("="*70)
print("ORIGINAL EMPLOYEES DATAFRAME (WITH DUPLICATES)")
print("="*70)
print(employees_df)
print(f"\nShape: {employees_df.shape}")
print(f"Total rows: {len(employees_df)}")

print("\n" + "="*70)
print("ORIGINAL SALES DATAFRAME (WITH DUPLICATES)")
print("="*70)
print(sales_df)
print(f"\nShape: {sales_df.shape}")
print(f"Total rows: {len(sales_df)}")

## Part 2: Detecting Duplicate Records

The `.duplicated()` method identifies which rows are duplicates of other rows.

### 2.1: Detecting Exact Duplicates (All Columns)

A row is an exact duplicate if all its values match another row exactly.

In [ ]:
print("DETECTING EXACT DUPLICATES - EMPLOYEES")
print("="*70)

# Check which rows are duplicates (compared to ALL previous rows)
duplicates_mask = employees_df.duplicated()

print("Boolean mask of duplicates (True = Duplicate):")
print(duplicates_mask.to_string())

print(f"\nTotal duplicate rows: {duplicates_mask.sum()}")
print(f"Unique rows: {(~duplicates_mask).sum()}")

print("\n" + "="*70)
print("DUPLICATE ROWS HIGHLIGHTED")
print("="*70)

# Show only the duplicate rows
duplicate_rows = employees_df[duplicates_mask]
print(f"\nShowing {len(duplicate_rows)} duplicate rows:")
print(duplicate_rows[["EmployeeID", "Name", "Department", "Salary"]])

### 2.2: Keep Parameter - Which Copy to Consider a Duplicate

The `keep` parameter controls which occurrence is marked as the duplicate.

In [ ]:
print("KEEP PARAMETER: WHICH COPY IS CONSIDERED A DUPLICATE?")
print("="*70)

print("\nExample: Alice appears 3 times (rows 0, 2, 7)")
alice_rows = employees_df[employees_df['Name'] == 'Alice']
print(alice_rows[["EmployeeID", "Name", "Department", "Salary"]])
print(f"Index positions: {alice_rows.index.tolist()}")

print("\n" + "-"*70)
print("\nkeep='first' - Mark copies AFTER first as duplicates (DEFAULT)")
dup_first = employees_df.duplicated(keep='first')
print(f"Total duplicates: {dup_first.sum()}")
print("Marked as duplicate:", dup_first.index[dup_first].tolist())
print("(Rows 2 and 7 marked as duplicates; row 0 kept)")

print("\n" + "-"*70)
print("\nkeep='last' - Mark copies BEFORE last as duplicates")
dup_last = employees_df.duplicated(keep='last')
print(f"Total duplicates: {dup_last.sum()}")
print("Marked as duplicate:", dup_last.index[dup_last].tolist())
print("(Rows 0 and 2 marked as duplicates; row 7 kept)")

print("\n" + "-"*70)
print("\nkeep=False - Mark ALL duplicates (none considered 'first')")
dup_all = employees_df.duplicated(keep=False)
print(f"Total duplicates: {dup_all.sum()}")
print("Marked as duplicate:", dup_all.index[dup_all].tolist())
print("(All 3 occurrences of Alice marked as duplicates)")

print("\n✓ When to use each:")
print("  keep='first': Keep earliest entry (default, most common)")
print("  keep='last': Keep most recent entry")
print("  keep=False: Both identify ALL duplicate instances")

### 2.3: Detecting Duplicates by Specific Columns

Sometimes you only want to detect duplicates based on certain columns.

In [ ]:
print("DETECTING DUPLICATES BY SPECIFIC COLUMNS")
print("="*70)

print("\nScenario: Check for duplicate employees by EmployeeID and Name")
print("(Ignore phone number differences)")

# Check if same employee appears more than once, regardless of phone
dup_by_name_id = employees_df.duplicated(subset=['EmployeeID', 'Name'])

print(f"\nTotal duplicates by (EmployeeID, Name): {dup_by_name_id.sum()}")
print(f"Duplicate indices: {dup_by_name_id.index[dup_by_name_id].tolist()}")

duplicate_employee_records = employees_df[dup_by_name_id]
print("\nDuplicate employee records:")
print(duplicate_employee_records[["EmployeeID", "Name", "Department", "PhoneNumber"]])

print("\n" + "="*70)
print("\nCompare: ALL columns vs SUBSET")
print("="*70)

dup_all_cols = employees_df.duplicated(keep=False)
dup_subset = employees_df.duplicated(subset=['EmployeeID', 'Name'], keep=False)

print(f"\nExact duplicates (all columns): {dup_all_cols.sum()}")
print(f"Duplicates by ID+Name (ignoring phone): {dup_subset.sum()}")
print(f"\nDifference: {dup_subset.sum() - dup_all_cols.sum()} rows differ only in non-key columns")

## Part 3: Inspecting Duplicate Records

Before removing duplicates, it's important to understand what makes them duplicates.

### 3.1: Find All Occurrences of Duplicate Values

Look at ALL instances (first + duplicates) to understand the data.

In [ ]:
print("INSPECTING ALL OCCURRENCES OF DUPLICATES")
print("="*70)

# Get ALL rows that appear more than once (including first occurrence)
# Using keep=False marks all duplicates
all_duplicate_instances = employees_df[employees_df.duplicated(keep=False)]

print(f"\nRows that appear multiple times: {len(all_duplicate_instances)}")
print("(This includes both the original and all copies)")
print("\nAll instances of duplicated records:")
print(all_duplicate_instances[["EmployeeID", "Name", "Department", "Salary", "PhoneNumber"]].sort_values('EmployeeID'))

print("\n" + "="*70)
print("Grouped view - See how many times each appears")
print("="*70)

# Count occurrences of each employee
occurrence_counts = employees_df.groupby(['EmployeeID', 'Name']).size().reset_index(name='Count')
occurrence_counts = occurrence_counts[occurrence_counts['Count'] > 1]

print("\nEmployees with duplicates:")
print(occurrence_counts.to_string(index=False))

### 3.2: Inspect Sales Data - Partial Duplicates

Some duplicates differ slightly in non-critical columns.

In [ ]:
print("INSPECTING SALES DATA - PARTIAL DUPLICATES")
print("="*70)

print("\nOriginal sales data:")
print(sales_df)

print("\n" + "="*70)
print("\nExact duplicates (all columns the same):")
exact_dup = sales_df.duplicated(keep=False)
exact_dup_rows = sales_df[exact_dup].sort_values(['Date', 'Product'])
print(f"Total exact duplicates: {exact_dup.sum()}")
if len(exact_dup_rows) > 0:
    print(exact_dup_rows)
else:
    print("(None)")

print("\n" + "="*70)
print("\nPartial duplicates (same Date, Product, Quantity, Price, Total):")
partial_dup = sales_df.duplicated(subset=['Date', 'Product', 'Quantity', 'Price', 'Total'], keep=False)
partial_dup_rows = sales_df[partial_dup].sort_values(['Date', 'Product'])
print(f"Total partial duplicates: {partial_dup.sum()}")
print(partial_dup_rows)

print("\nNote: Rows 2 and 12 (Phone sales) differ ONLY in Region")
print("This is a 'partial duplicate' - same transaction, different field")

## Part 4: Removing Duplicate Records

The `.drop_duplicates()` method removes duplicate rows from the DataFrame.

### 4.1: Remove Exact Duplicates (All Columns)

Keeping the first occurrence of each unique row.

In [ ]:
print("REMOVING EXACT DUPLICATES - EMPLOYEES")
print("="*70)

# Original state
print(f"Original shape: {employees_df.shape}")
print(f"Original rows: {len(employees_df)}")
print(f"Original columns: {len(employees_df.columns)}")

# Remove duplicates (keeps first occurrence)
employees_dedup = employees_df.drop_duplicates()

print(f"\nAfter drop_duplicates(): {employees_dedup.shape}")
print(f"Remaining rows: {len(employees_dedup)}")
print(f"Remaining columns: {len(employees_dedup.columns)}")

# Impact
rows_removed = len(employees_df) - len(employees_dedup)
pct_removed = (rows_removed / len(employees_df)) * 100

print(f"\nIMPACT:")
print(f"  Rows removed: {rows_removed} ({pct_removed:.1f}%)")
print(f"  Rows kept: {len(employees_dedup)}")

print("\nCleaned employees:")
print(employees_dedup[["EmployeeID", "Name", "Department", "Salary"]].to_string())

### 4.2: Remove Duplicates by Specific Columns

Keep first occurrence when compared by subset of columns.

In [ ]:
print("REMOVING DUPLICATES BY SPECIFIC COLUMNS")
print("="*70)

print("\nScenario: Remove duplicate employees by (EmployeeID, Name)")
print("(Ignore phone number differences)")

# Original state
print(f"\nOriginal rows: {len(employees_df)}")

# Remove duplicates considering only EmployeeID and Name
employees_dedup_subset = employees_df.drop_duplicates(subset=['EmployeeID', 'Name'])

print(f"After drop_duplicates(subset=['EmployeeID', 'Name']): {len(employees_dedup_subset)} rows")

# Impact
rows_removed = len(employees_df) - len(employees_dedup_subset)
pct_removed = (rows_removed / len(employees_df)) * 100

print(f"\nIMPACT:")
print(f"  Rows removed: {rows_removed} ({pct_removed:.1f}%)")
print(f"  Rows kept: {len(employees_dedup_subset)}")

print("\nResult (all columns shown):")
print(employees_dedup_subset.to_string())

print("\nCompare with exact duplicate removal:")
print(f"  Exact duplicates removed: {len(employees_df) - len(employees_dedup)} rows")
print(f"  Partial duplicates removed: {len(employees_df) - len(employees_dedup_subset)} rows")
print(f"  Difference: {len(employees_dedup_subset) - len(employees_dedup)} rows differ in non-key columns")

### 4.3: Keep Last vs Keep First

Choose which occurrence to keep when removing duplicates.

In [ ]:
print("CHOOSING WHICH OCCURRENCE TO KEEP")
print("="*70)

# Show example: Bob appears in rows 1, 4, 9
print("\nExample: Bob appears 3 times:")
bob_rows = employees_df[employees_df['Name'] == 'Bob']
print(bob_rows[["EmployeeID", "Name", "Salary", "PhoneNumber"]])
print("Row indices:", bob_rows.index.tolist())

print(f"\nVersion 1: keep='first' (default)")
keep_first = employees_df.drop_duplicates(subset=['EmployeeID', 'Name'], keep='first')
bob_kept = keep_first[keep_first['Name'] == 'Bob']
print("Bob record kept:")
print(bob_kept[["EmployeeID", "Name", "Salary", "PhoneNumber"]].to_string())

print(f"\nVersion 2: keep='last'")
keep_last = employees_df.drop_duplicates(subset=['EmployeeID', 'Name'], keep='last')
bob_kept = keep_last[keep_last['Name'] == 'Bob']
print("Bob record kept:")
print(bob_kept[["EmployeeID", "Name", "Salary", "PhoneNumber"]].to_string())

print("\nNote: Kept row differs! Row 1 has phone 555-0102, Row 9 has 555-9999")

print("\n✓ When to use each:")
print("  keep='first': Most common - keep earliest entry")
print("  keep='last': Keep most recent entry (newer records)")

### 4.4: Remove Duplicates from Sales Data

Handling duplicate transactions in sales data.

In [ ]:
print("REMOVING DUPLICATES - SALES DATA")
print("="*70)

print(f"\nOriginal shape: {sales_df.shape}")
print(f"Original rows: {len(sales_df)}")

print("\nOriginal data:")
print(sales_df)

# Remove exact duplicates
sales_dedup = sales_df.drop_duplicates()

print(f"\nAfter drop_duplicates(): {sales_dedup.shape}")
print(f"Rows removed: {len(sales_df) - len(sales_dedup)}")

print("\nCleaned sales data:")
print(sales_dedup[["Date", "Product", "Quantity", "Total", "Region"]])

print("\n" + "="*70)
print("Strategic removal: Duplicates by transaction core columns")
print("="*70)

# What if Region differences matter?
# Remove duplicates ONLY considering core transaction columns
sales_dedup_core = sales_df.drop_duplicates(subset=['Date', 'Product', 'Quantity', 'Price', 'Total'])

print(f"\nRemoving duplicates by (Date, Product, Quantity, Price, Total):")
print(f"Rows after deduplication: {len(sales_dedup_core)}")
print(f"Rows removed: {len(sales_df) - len(sales_dedup_core)}")

print("\nResult:")
print(sales_dedup_core)

## Part 5: Verification - Confirming Duplicates Are Removed

Always verify that deduplication was successful.

### 5.1: Verify No Duplicates Remain

In [ ]:
print("VERIFICATION: CONFIRMING DUPLICATES ARE REMOVED")
print("="*70)

print("\n1. CHECK DUPLICATED COUNT")
print("-" * 70)

# Original
original_duplicates = employees_df.duplicated().sum()
print(f"Original employees: {original_duplicates} duplicate rows")

# After deduplication
dedup_duplicates = employees_dedup.duplicated().sum()
print(f"After drop_duplicates(): {dedup_duplicates} duplicate rows")
print(f"Status: {'✓ VERIFIED' if dedup_duplicates == 0 else '✗ FAILED'}")

print("\n2. SHAPE COMPARISON")
print("-" * 70)
print(f"Original shape: {employees_df.shape}")
print(f"After dedup:   {employees_dedup.shape}")
print(f"Rows lost: {employees_df.shape[0] - employees_dedup.shape[0]}")
print(f"Columns unchanged: {employees_df.shape[1] == employees_dedup.shape[1]}")

print("\n3. UNIQUENESS CHECK")
print("-" * 70)

# Check unique counts
original_unique_employees = employees_df.groupby(['EmployeeID', 'Name']).size().shape[0]
dedup_unique_employees = employees_dedup.groupby(['EmployeeID', 'Name']).size().shape[0]

print(f"Unique (EmployeeID, Name) combinations:")
print(f"  Original: {original_unique_employees}")
print(f"  After dedup: {dedup_unique_employees}")
print(f"  All unique: {dedup_unique_employees == len(employees_dedup)}")

print("\n4. SPOT-CHECK: Sample of cleaned data")
print("-" * 70)
print(employees_dedup[["EmployeeID", "Name", "Department"]].head(10).to_string())

### 5.2: Why Verification Matters

In [ ]:
print("\nWHY VERIFICATION IS CRITICAL")
print("="*70)

print("\n❌ PITFALL: We removed 5 rows. Did we remove the RIGHT duplicates?")
print("\nWithout verification, we might:")
print("  - Keep an outdated record when we wanted the latest")
print("  - Accidentally keep incomplete data")
print("  - Lose important information from the deduplication")
print("  - Not realize the deduplication didn't work")

print("\n✅ VERIFICATION SAFEGUARDS:")
print("  1. Count duplicates BEFORE and AFTER")
print(f"     Before: {original_duplicates}, After: {dedup_duplicates}")
print("     Action: ✓ Confirmed removed")

print("\n  2. Check shape didn't break")
print(f"     Before: {employees_df.shape}, After: {employees_dedup.shape}")
print("     Action: ✓ Confirmed same columns")

print("\n  3. Verify uniqueness of key columns")
print(f"     All rows unique by (ID, Name): {dedup_unique_employees == len(employees_dedup)}")
print("     Action: ✓ Confirmed")

print("\n  4. Spot-check data reasonableness")
print(f"     Employees in dedup: {len(employees_dedup)}")
print(f"     Unique IDs: {employees_dedup['EmployeeID'].nunique()}")
print("     Action: ✓ Confirmed reasonable")

print("\n  5. Document what was removed")
print(f"     Removed {original_duplicates} duplicate entries")
print(f"     Kept {len(employees_dedup)} unique employee records")
print("     Action: ✓ Documented")

## Part 6: Scenario Analysis - Deciding Which Duplicates to Keep

When duplicates differ slightly, how do you decide which record to keep?

### 6.1: Exact Duplicates - Clear Decision

### 6.2: Partial Duplicates - Intentional Decision Required

In [ ]:
print("SCENARIO 2: PARTIAL DUPLICATES (Differ in non-critical column)")
print("="*70)

print("\nExample: Bob's salary updated from 60000 to 65000")
bob_records = employees_df[employees_df['EmployeeID'] == 1002]
print(bob_records[["EmployeeID", "Name", "Salary", "PhoneNumber"]].to_string())
print("\nRows 1, 4, 9: Same employee, but different records")
print("  Row 1: Salary 60000, Phone 555-0102 (original)")
print("  Row 4: Salary 60000, Phone 555-0102 (exact duplicate)")
print("  Row 9: Salary 60000, Phone 555-9999 (different phone!)")

print("\n" + "="*70)
print("DECISION OPTIONS:")
print("="*70)

print("\nOPTION A: Remove ALL duplicates by (EmployeeID, Name)")
print("-" * 70)
print("  .drop_duplicates(subset=['EmployeeID', 'Name'], keep='first')")
print("  Result: Keep row 1 (first occurrence)")
print("  Phone: 555-0102")
print("\n  Context: Choose FIRST entry")
print("  Best when: Historical accuracy matters (when did they start?)")
print("  Risk: May lose updated information")

print("\nOPTION B: Remove ALL duplicates, keep LAST")
print("-" * 70)
print("  .drop_duplicates(subset=['EmployeeID', 'Name'], keep='last')")
print("  Result: Keep row 9 (most recent)")
print("  Phone: 555-9999")
print("\n  Context: Choose LATEST record")
print("  Best when: Current information matters (what's their phone now?)")
print("  Risk: Might lose historical context")

print("\nOPTION C: Keep only EXACT duplicates, investigate differences")
print("-" * 70)
print("  .drop_duplicates(keep='first')  # All columns must match")
print("  Result: Removes rows 1 & 4 (exact), keeps 9 (different phone)")
print("\n  Context: Only remove IDENTICAL records")
print("  Best when: Phone differences are meaningful")
print("  Requires: Manual review of why phone changed")

print("\n" + "="*70)
print("DECISION FRAMEWORK")
print("="*70)
print("\n1. Is the difference meaningful?")
print("   YES → Investigate further (may not be a duplicate)")
print("   NO → Remove duplicate")

print("\n2. What changed?")
print("   Non-critical field (phone) → Can choose either")
print("   Critical field (salary) → Need business logic")

print("\n3. Time context?")
print("   keep='first': Preserve original/baseline")
print("   keep='last': Keep most current")

print("\n4. Uncertainty?")
print("   → Flag for manual review")
print("   → Keep FIRST and mark for investigation")
print("   → Document your decision")

### 6.3: Business Context Matters

In [ ]:
print("WHY BUSINESS CONTEXT MATTERS")
print("="*70)

print("\nSame data, DIFFERENT deduplication by context:")
print("-" * 70)

print("\nScenario A: FINANCIAL CONTEXT")
print("Question: What is each employee's current salary for payroll?")
print("Decision: Keep LAST record (most recent salary)")
print("Reasoning: Payroll must reflect current rates")
print("Method: keep='last'")

print("\nScenario B: HISTORICAL CONTEXT")
print("Question: When did each employee start with us?")
print("Decision: Keep FIRST record (hire date reference)")
print("Reasoning: Need baseline to calculate tenure")
print("Method: keep='first'")

print("\nScenario C: CONTACT CONTEXT")
print("Question: What's the most reliable phone number?")
print("Decision: Investigate BOTH records")
print("Reasoning: Different phone might mean system error OR legitimate update")
print("Method: keep=False (flag all instances for review)")

print("\nScenario D: INTEGRITY CONTEXT")
print("Question: Which record is most complete/accurate?")
print("Decision: Compare data quality")
print("Reasoning: Not about which is first/last, but which is RIGHT")
print("Method: Custom logic (not just drop_duplicates)")

print("\n" + "="*70)
print("IMPACT ON ANALYSIS")
print("="*70)

# Show impact
print("\nChoosing FIRST vs LAST changes the results:")
print("\nkeep='first': Keep Bob's salary as 60000")
print("  Total payroll: $", employees_dedup[employees_dedup['Name'] == 'Bob']['Salary'].sum() + 65000)
print("  (Bob counted as 60000)")

print("\nkeep='last': Keep Bob's salary as 60000 (row 9 same salary, just phone different!)")
print("  Total payroll: Same $")
print("  But phone number changes interpretation!")

print("\nThe point: Different deduplication → Different results")
print("           → Intentional decision IS important")

## Part 7: Implementation Checklist

A practical checklist for deduplicating your own data.

In [ ]:
print("DEDUPLICATION IMPLEMENTATION CHECKLIST")
print("="*70)

print("\n☐ STEP 1: DETECT & INSPECT")
print("  ☐ Load data and check for duplicates: df.duplicated().sum()")
print("  ☐ Look at examples: df[df.duplicated(keep=False)].sort_values(...)")
print("  ☐ Check if exact duplicates or partial: Try subset parameter")
print("  ☐ Understand WHY duplicates exist (error? update? merge artifact?)")

print("\n☐ STEP 2: ANALYZE PATTERNS")
print("  ☐ How many duplicate rows? (Count and percentage)")
print("  ☐ Which columns identify a unique record?")
print("  ☐ Do differences in duplicates matter (critical vs non-critical)?")
print("  ☐ Time component? (Should we keep first or last?)")

print("\n☐ STEP 3: DECIDE STRATEGY")
print("  ☐ Will you remove by all columns or a subset?")
print("  ☐ Keep='first' or 'last' or 'False' (investigate all)?")
print("  ☐ Document your reasoning for this choice")
print("  ☐ Consider business impact of this decision")

print("\n☐ STEP 4: IMPLEMENT & VERIFY")
print("  ☐ Create a copy if unsure: df_dedup = df.drop_duplicates(...)")
print("  ☐ Immediately verify: df_dedup.duplicated().sum() == 0")
print("  ☐ Check shape loss is expected: len(df_dedup) vs len(df)")
print("  ☐ Spot-check the removed rows match expectations")

print("\n☐ STEP 5: DOCUMENT & COMMUNICATE")
print("  ☐ Add comments explaining the deduplication logic")
print("  ☐ Record: Original rows, rows removed, final rows")
print("  ☐ Include reasoning: Why this deduplication method?")
print("  ☐ Flag any uncertain/borderline duplicates for manual review")

print("\n" + "="*70)
print("EXAMPLE CODE")
print("="*70)

print("\n# Step 1: Detect")
print("print('Duplicates:', df.duplicated().sum())")
print("print(df[df.duplicated(keep=False)].sort_values('EmployeeID'))")

print("\n# Step 2: Analyze")
print("# Duplicates are exact copies (all columns)")
print("# No time component, so keep='first' is reasonable")

print("\n# Step 3: Decide")
print("# Remove all exact duplicates, keeping first occurrence")

print("\n# Step 4: Implement")
print("df_dedup = df.drop_duplicates(keep='first')")
print("assert df_dedup.duplicated().sum() == 0  # Verify")
print(f"print(f'Removed {{len(df) - len(df_dedup)}} duplicate rows')")

print("\n# Step 5: Document")
print("# Removed 3 exact duplicate employee records")
print("# Kept first occurrence of each employee")
print("# Final dataset: 10 unique employees")

## Summary: Key Takeaways

### Detection Methods
- **`.duplicated()`**: Returns Boolean array; True = row is a duplicate
- **`.duplicated(subset=[...])`**: Check duplicates by specific columns only
- **`keep='first'`**: Mark copies AFTER the first as duplicates (default)
- **`keep='last'`**: Mark copies BEFORE the last as duplicates
- **`keep=False`**: Mark ALL instances as duplicates (including first)

### Removal Methods
- **`.drop_duplicates()`**: Remove rows where all values are duplicated
- **`.drop_duplicates(subset=[...])`**: Remove based on specific columns
- **`keep='first'`**: Keep first occurrence, remove subsequent copies
- **`keep='last'`**: Keep last occurrence, remove prior copies

### Exact vs Partial Duplicates
- **Exact**: All columns identical → Clear duplicates, safe to remove
- **Partial**: Columns differ in non-critical fields → Requires decision

### Decision Factors
- **Business context**: What is this data used for?
- **Time sensitivity**: Should we keep first (baseline) or last (current)?
- **Data quality**: Which record is more complete/accurate?
- **Impact**: How does deduplication affect downstream analysis?

### Verification Checklist
1. Count duplicates before and after
2. Verify no duplicates remain: `.duplicated().sum() == 0`
3. Check shape didn't break columns
4. Spot-check removed data matches expectations
5. Document why this deduplication approach was chosen